# Bronze Layer

Ingests the raw Synthea healthcare tables from the Snowflake `STAGING` schema, stamps them with ingestion metadata, and lands them unmodified (structurally) into the Snowflake `BRONZE` schema.

This notebook is self-contained: it can be run top-to-bottom on its own kernel without any dependency on another notebook.

## 1. Imports

In [ ]:
%pip install -U pyspark==4.0.4

In [ ]:
import os

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit

print("PySpark:", pyspark.__version__)

## 2. Configuration

Snowflake connection options and the list of source tables to migrate from `STAGING` into `BRONZE`. Centralized here so it is only defined once for the whole notebook.

In [ ]:
# Required Spark <-> Snowflake connector packages
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "net.snowflake:snowflake-jdbc:4.0.2,"
    "net.snowflake:spark-snowflake_2.13:3.2.1-spark_4.0 "
    "pyspark-shell"
)

# Snowflake connection options
# NOTE: preserved exactly as configured in the original project notebook.
sfOptions = {
    "sfURL": "zj90931.eu-central-2.aws.snowflakecomputing.com",
    "sfUser": "ahmedSami",
    "sfPassword": os.getenv("SNOWFLAKE_PASSWORD", ""),
    "sfDatabase": "HEALTHCARE_DB",
    "sfWarehouse": "HEALTHCARE_WH",
    "sfRole": "ACCOUNTADMIN",
}

SNOWFLAKE_FORMAT = "net.snowflake.spark.snowflake"

# Source tables staged in Snowflake that this notebook ingests into BRONZE
SOURCE_TABLES = [
    "PATIENTS",
    "ENCOUNTERS",
    "OBSERVATIONS",
    "CONDITIONS",
    "MEDICATIONS",
    "ALLERGIES",
    "PROCEDURES",
    "IMMUNIZATIONS",
    "CAREPLANS",
]

print(f"Number of source tables: {len(SOURCE_TABLES)}")

## 3. Spark Session

In [ ]:
spark = SparkSession.builder \
    .appName("Bronze_Layer_Processing") \
    .getOrCreate()

print("Spark:", spark.version)
print("Scala:", spark.sparkContext._jvm.scala.util.Properties.versionNumberString())

## 4. Source Data → Bronze Tables

For every source table: read the raw data from `STAGING`, add minimal ingestion-level metadata (`INGESTION_TIMESTAMP`, `SOURCE_SYSTEM`), and write it to `BRONZE` as `<TABLE>_BRONZE`. No business transformations happen here — Bronze preserves the raw source data as closely as possible.

In [ ]:
def read_staging_table(table_name):
    """Read a raw table from the Snowflake STAGING schema."""
    options = dict(sfOptions)
    options["sfSchema"] = "STAGING"
    return (
        spark.read
        .format(SNOWFLAKE_FORMAT)
        .options(**options)
        .option("dbtable", table_name)
        .load()
    )


def write_bronze_table(df, table_name):
    """Write a DataFrame to the Snowflake BRONZE schema."""
    options = dict(sfOptions)
    options["sfSchema"] = "BRONZE"
    (
        df.write
        .format(SNOWFLAKE_FORMAT)
        .options(**options)
        .option("dbtable", table_name)
        .mode("overwrite")
        .save()
    )

In [ ]:
print("Starting Data Migration: STAGING -> BRONZE")

for table in SOURCE_TABLES:
    print(f"\nProcessing Table: {table}")

    # 1. Read from STAGING
    raw_df = read_staging_table(table)

    # 2. Add Bronze ingestion metadata
    bronze_df = raw_df \
        .withColumn("INGESTION_TIMESTAMP", current_timestamp()) \
        .withColumn("SOURCE_SYSTEM", lit("SYNTHEA_BATCH_1"))

    # 3. Write to BRONZE
    bronze_table_name = f"{table}_BRONZE"
    write_bronze_table(bronze_df, bronze_table_name)

    print(f"{table} successfully saved to BRONZE as {bronze_table_name}")

print(f"\nAll {len(SOURCE_TABLES)} tables successfully migrated to BRONZE!")

## 5. Validation

Confirm every Bronze table was created successfully and has the expected ingestion metadata columns.

In [ ]:
print("Validating BRONZE tables...\n")

for table in SOURCE_TABLES:
    bronze_table_name = f"{table}_BRONZE"
    try:
        options = dict(sfOptions)
        options["sfSchema"] = "BRONZE"
        df = (
            spark.read
            .format(SNOWFLAKE_FORMAT)
            .options(**options)
            .option("dbtable", bronze_table_name)
            .load()
        )
        row_count = df.count()
        has_metadata = {"INGESTION_TIMESTAMP", "SOURCE_SYSTEM"}.issubset(set(df.columns))
        status = "OK" if has_metadata else "MISSING METADATA COLUMNS"
        print(f"[{status}] {bronze_table_name}: {row_count:,} rows, {len(df.columns)} columns")
    except Exception as e:
        print(f"[FAILED] {bronze_table_name}: {e}")